In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df1 = pd.read_csv('../data/letters_2021.csv')
df2 = pd.read_csv('../data/sentence_sets_trimmed.csv', encoding='mac-roman')
df2 = df2[df2['full_text_tokens'] > 10]

In [4]:
raw_pairs = {}
with open("../data/gendered_term_list.txt", "r", encoding="utf-8") as f:
    for line in f:
        key, value = line.strip().split("\t")
        raw_pairs[key] = value

In [19]:
# raw_pairs

{'masculine': 'feminine',
 'masculinely': 'femininely',
 'masculinity': 'femininity',
 'masculism': 'feminism',
 'masculinist': 'feminist',
 'patriarch': 'matriarch',
 'patriarchal': 'matriarchal',
 'male': 'female',
 'males': 'females',
 'maleness': 'femaleness',
 'boy': 'girl',
 'boys': 'girls',
 'gamine': 'girly',
 'boyhood': 'girlhood',
 'boyish': 'girlish',
 'boyishly': 'girlishly',
 'boyishness': 'girlishness',
 'man': 'woman',
 'men': 'women',
 'manhood': 'womanhood',
 'manliness': 'womanliness',
 'manly': 'womanly',
 'virile': 'uxorially',
 'virility': 'milkiness',
 'sir': 'madam',
 'sirs': 'madams',
 'widower': 'widow',
 'widowers': 'widows',
 'mr': 'ms',
 'mister': 'ms',
 'gentleman': 'damsel',
 'gentlemen': 'damsels',
 'gentlemanlike': 'ladylike',
 'lord': 'lady',
 'lords': 'ladies',
 'groom': 'bride',
 'grooms': 'brides',
 'bridegroom': 'bride',
 'bridegrooms': 'brides',
 'best man': 'bridesmaid',
 'best men': 'bridesmaids',
 'fiance': 'fiancee',
 'fiancé': 'fiancée',
 'fia

# Preprocess Data

In [5]:
degender_mapping = {
    rf"(?:^|\b|[^\w\s])(?P<token>{re.escape(k)})(?P<suffix>'s|’s)?(?=\b|[^\w\s]|$)": v
    for k, v in raw_pairs.items()
}

In [6]:
degender_mapping

{"(?:^|\\b|[^\\w\\s])(?P<token>masculine)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'feminine',
 "(?:^|\\b|[^\\w\\s])(?P<token>masculinely)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'femininely',
 "(?:^|\\b|[^\\w\\s])(?P<token>masculinity)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'femininity',
 "(?:^|\\b|[^\\w\\s])(?P<token>masculism)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'feminism',
 "(?:^|\\b|[^\\w\\s])(?P<token>masculinist)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'feminist',
 "(?:^|\\b|[^\\w\\s])(?P<token>patriarch)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'matriarch',
 "(?:^|\\b|[^\\w\\s])(?P<token>patriarchal)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'matriarchal',
 "(?:^|\\b|[^\\w\\s])(?P<token>male)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'female',
 "(?:^|\\b|[^\\w\\s])(?P<token>males)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'females',
 "(?:^|\\b|[^\\w\\s])(?P<token>maleness)(?P<suffix>'s|’s)?(?=\\b|[^\\w\\s]|$)": 'femaleness',
 "(?:^|\\b|[^\\w\\s])(?P<token>boy)(?P<suffix>'s|’s)?(?=\\b|[^\\w

In [7]:
df1 = df1[['LETTERTEXT', 'LETTER_GENDER']]
df1 = df1.rename(columns={'LETTERTEXT':'full_text', 'LETTER_GENDER':'label'})
df1['full_text'] = df1['full_text'].str.lower()

In [8]:
def apply_degendering(text):
    for pattern, replacement in degender_mapping.items():
        def repl(match):
            token = match.group("token")
            suffix = match.group("suffix") or ""
            replacement_base = raw_pairs.get(token.lower(), token)
            return match.group(0).replace(token + suffix, replacement_base + suffix)
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    return text

In [9]:
df1['full_text'] = df1['full_text'].apply(apply_degendering)

### Second Dataset

In [10]:
df2 = df2[['full_text', 'applicant_gender']]
df2 = df2.rename(columns={'applicant_gender':'label'})
df2['full_text'] = df2['full_text'].str.lower()

In [11]:
df2['full_text'] = df2['full_text'].apply(apply_degendering)

# Combine Data

In [12]:
apply_degendering("he is the husband's brother's uncle's actor and fireman. he is a male and is a father of three sons. (he'll) be a great doctor. his helped him, and himself.")

"she is the wife's sister's aunt's actress and firewoman. she is a female and is a mother of three daughters. (she'll) be a great doctor. her helped her, and herself."

In [13]:
df = pd.concat([df1, df2], ignore_index=True)

In [14]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [15]:
df['label'] = df['label'].replace(gender_label_mapping)

<ipython-input-15-cc45885305bc>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['label'] = df['label'].replace(gender_label_mapping)


In [17]:
df

,full_text,label
0,it is my pleasure to write a letter of recomme...,0
1,i am pleased to highly recommend identifier fo...,0
2,i am writing this letter in support of identif...,0
3,identifier identifier recently completed an an...,0
4,it is my pleasure to recommend dr. identifier ...,0
...,...,...
8982,it is with pleasure that i recommend first_nam...,0
8983,we are very pleased to write this letter based...,0
8984,i am writing this letter of recommendation for...,1
8985,it is my pleasure to write first_name support ...,1


In [18]:
# df.to_csv('../data/combined_letters_degendered.csv', index=False)

# Create Gendered Mapping Dictionary

In [20]:
raw_pairs

{'masculine': 'feminine',
 'masculinely': 'femininely',
 'masculinity': 'femininity',
 'masculism': 'feminism',
 'masculinist': 'feminist',
 'patriarch': 'matriarch',
 'patriarchal': 'matriarchal',
 'male': 'female',
 'males': 'females',
 'maleness': 'femaleness',
 'boy': 'girl',
 'boys': 'girls',
 'gamine': 'girly',
 'boyhood': 'girlhood',
 'boyish': 'girlish',
 'boyishly': 'girlishly',
 'boyishness': 'girlishness',
 'man': 'woman',
 'men': 'women',
 'manhood': 'womanhood',
 'manliness': 'womanliness',
 'manly': 'womanly',
 'virile': 'uxorially',
 'virility': 'milkiness',
 'sir': 'madam',
 'sirs': 'madams',
 'widower': 'widow',
 'widowers': 'widows',
 'mr': 'ms',
 'mister': 'ms',
 'gentleman': 'damsel',
 'gentlemen': 'damsels',
 'gentlemanlike': 'ladylike',
 'lord': 'lady',
 'lords': 'ladies',
 'groom': 'bride',
 'grooms': 'brides',
 'bridegroom': 'bride',
 'bridegrooms': 'brides',
 'best man': 'bridesmaid',
 'best men': 'bridesmaids',
 'fiance': 'fiancee',
 'fiancé': 'fiancée',
 'fia

In [21]:
raw_pairs_df = pd.DataFrame(list(raw_pairs.items()), columns=['male term', 'female term'])

In [ ]:
# raw_pairs_df.to_csv('../data/gendered_term_mapping.csv', index=False)